In [ ]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_community.document_loaders import UnstructuredFileLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.storage import LocalFileStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough


llm = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0.1
)

cache_dir = LocalFileStore("./.cache/")



splitter = CharacterTextSplitter.from_tiktoken_encoder(
    model_name="gpt-5-nano",
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

loader = UnstructuredFileLoader("./files/chapter_one.docx")

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cache_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings, cache_dir
)

vectorstore = FAISS.from_documents(docs, cache_embeddings)

retrivar = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant. Asnwer questions using only the following context.
     If you don't know the answer just say you don't know,
     don't make it up:\n\n{context}"""),
    ("human", "{question}")
])

chain = ({
    "context":retrivar(), 
    "question": RunnablePassthrough(), 
    } 
    | prompt 
    | llm
)
chain.invoke("Describ Victory Mansions")

/home/nomadcoders/documents/studydir/nomadcoders/FULLSTACK-GPT/env/lib/python3.12/site-packages/langchain_classic/chains/combine_documents/map_rerank.py:190: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  results = self.llm_chain.apply_and_parse(


'Helpful Answer: Victory Mansions is a dreary, austerely maintained block of flats. The lift rarely works, and at daylight hours the electricity is cut off as part of the economy drive for Hate Week. The hallway smells of boiled cabbage and old rag mats. A large poster of an enormous face (Big Brother) dominates the wall, and its eyes seem to follow you as you move. Winston climbs seven flights up because the lift is out. Inside the flat, a telescreen is mounted on the wall, and a voice reads out figures related to production, underscoring the ambience of surveillance and control.'